# Variance-narrowing reproduction notebook (PI-removal, pooled-variance and session-SD constructions)

Recomputes **every variance-narrowing number** quoted for the publication from the frozen CSVs: the pooled-variance curve estimates (exp5-prescreen and Exp4) with and without the PI, the PI-as-one-person decomposition, the Part 7 per-W ratio table, and the prereg-style per-session SD construction. It does **not** recompute the paired ΔH_RS contrast (already exists for Exp4).
It ends with a PASS/FAIL table comparing each recomputed number with the published value.

**Conventions (do not change without saying so)**
- *PI* = all 10 PI/test Firebase UIDs, treated as **one person**. Exp4 has only `y612…`.
- *Without PI* = removed from the **Human arm only. Baseline is never trimmed** (Baseline is automated runs from the PI's laptop).
- *Pooled variance* statistic (Part 17b): mean over W=3/5/10 of log(pooled variance of windowed Pearson r, Human / Baseline), all windows pooled across sessions; participant → session cluster bootstrap for Human, session bootstrap for Baseline.
- *Session SD* statistic (prereg-style): SD(r) within each session (ddof=1, at least 2 windows); Human = mean over participants of each participant's mean session SD; Baseline = mean over sessions; W=5/10/15/20.
- Everything is **exploratory**. Only compare with-PI vs without-PI rows **within the same snapshot**.

**Inputs:** each file is read from a local copy if present (`../data` = `Study 1b/data`, the repo-level `data/`, the working directory, the Colab path, or `QART_DATA_DIR`); otherwise it is **downloaded from the repo** the way the `Exp4_Notebook1_Zen` notebooks fetch theirs: exp5-prescreen files from `.../main/Study%201b/data`, Exp4 files from `.../main/data`. Needed: `Frozen_Exp5Prescreen_{Blocks,Sessions}_2026-07-25.csv`, `..._2026-09-23.csv`, `Frozen_Blocks_2026-02-10_195735.csv`, `Frozen_Sessions_2026-02-10_195735.csv`. Optional: `..._2026-08-26.csv` (PI share only) and `..._2026-09-30.csv` (run-mode section). sha256 of each input is printed and checked.
**Runtime:** about 15–25 minutes at the published bootstrap sizes. Set `QUICK = True` for a fast smoke test (numbers will not match).
Companion index of all numbers and superseded values: `experiments/exp5-prescreen/RESULTS_INDEX_2026-09-30.md`.

In [1]:
import os, ast, hashlib, warnings, itertools, time
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')
T0 = time.time()

QUICK = False            # True = 500 bootstrap draws (smoke test only; will NOT match published numbers)
RUN_MODE_CHECK = False   # optional section 8 (needs the 2026-09-30 exp5 files; slow)
N_BOOT_POOLED = 500 if QUICK else 5000    # curve-level estimate (Part 17b uses N_BOOT_DEFAULT // 2 = 5000)
N_BOOT_RATIO  = 500 if QUICK else 10000   # Part 7 per-W ratios (N_BOOT_DEFAULT)
N_BOOT_SD     = 500 if QUICK else 10000   # session-SD construction
SEED_POOLED_EXP5, SEED_POOLED_EXP4, SEED_SD = 20260904, 20260903, 20260930

PI_IDS = {  # all PI/test accounts, treated as ONE person
    'y612YV0ACNSi8gs6TnC3QIueQi23', 'wvToPbdcnOhWXWmVQHA2pBinMb43', 'sum3kpNAYQYkSJUbXq700eoxAz52',
    'aYuCWBAcACaz6v1lBdQfjYfoQlp1', 'NLPGSuLn3VhsK7d5myVBSTX651t2', 'aYQL5LQbYUfvmcahficaM6fz2032',
    'D9RxmtooI1RpT4FtqUGo9YSOQ5C2', 'NUu3vPWCY9PglydsliaNQEx3AVM2', 'D4uQnMh6ROZBh1w7lmIryOScGM93',
    '03XJhnvMIddSPb7o1xRSXQ4VY1X2'}
Y612 = 'y612YV0ACNSi8gs6TnC3QIueQi23'
PI_TAG = 'PI_PERSON'

SEARCH_DIRS = [Path(os.environ['QART_DATA_DIR'])] if os.environ.get('QART_DATA_DIR') else []
SEARCH_DIRS += [Path.cwd(), Path.cwd() / 'data', Path.cwd().parent / 'data',            # Study 1b/analysis -> Study 1b/data
                Path.cwd().parent.parent / 'data',                                       # repo-level data/ (Exp4 frozen files)
                Path.cwd() / 'notebooks', Path('/content/drive/MyDrive/QART_Project')]
import urllib.request
DATA_BASES = ["https://raw.githubusercontent.com/catboxer/Beyond-The-Mean/main/Study%201b/data",   # exp5-prescreen frozen files
              "https://raw.githubusercontent.com/catboxer/Beyond-The-Mean/main/data"]               # Exp4 frozen files (same folder the Exp4_Notebook1_Zen notebooks use)
def find(name, required=True):
    # 1) a local copy (QART_DATA_DIR, working dir, notebooks/, Colab Drive); 2) otherwise download it from DATA_BASE into the
    # working directory, exactly as the Exp4_Notebook1_Zen notebooks do (urllib.request.urlretrieve).
    for d in SEARCH_DIRS:
        if (d / name).is_file(): return str(d / name)
    for d in SEARCH_DIRS:
        if d.is_dir():
            hits = sorted(d.rglob(name))
            if hits: return str(hits[0])
    err = None
    for base in DATA_BASES:
        try:
            urllib.request.urlretrieve(f"{base}/{name}", name)
            print(f"downloaded {name} from {base}")
            return name
        except Exception as e:
            err = e
            if os.path.exists(name) and os.path.getsize(name) == 0: os.remove(name)
    if required:
        raise FileNotFoundError(f"{name}: not found locally ({[str(d) for d in SEARCH_DIRS]}) and not downloadable from {DATA_BASES} ({err}). "
                                f"Put the file in ../data or the working directory, set QART_DATA_DIR, or upload it to the repo's Study 1b/data folder.")
    return None

EXPECTED_SHA = {
 'Frozen_Exp5Prescreen_Blocks_2026-07-25.csv': 'bd061f5689188428b464dfab3c847491a4d731a465728facb1f989b08b3d1b42',
 'Frozen_Exp5Prescreen_Sessions_2026-07-25.csv': '00bac7096bb716e07c28c7363a5facea3e702b8d245943bf2ffc20ae41081dc3',
 'Frozen_Exp5Prescreen_Blocks_2026-09-23.csv': 'ab995877735a27097b5ad28c87113bf879e35d1537bae047058bd61d82b6a863',
 'Frozen_Exp5Prescreen_Sessions_2026-09-23.csv': '3850a6ed8dc90b0bcff6cee961b4fbb84953f0588b17acdaa2e24dd3388996ea',
 'Frozen_Blocks_2026-02-10_195735.csv': '3328875750873c704abc47c872064c4878cb0f9d7f621a95dfdcc393b9f0b529',
 'Frozen_Sessions_2026-02-10_195735.csv': '6b54ae50ba65bcfaf2fe5a4f452765540dee2e1a3afd2da524418e5bc97f83b9',
 'Frozen_Exp5Prescreen_Sessions_2026-09-30.csv': 'b3454974589158b102c03451b2a7096e605263a62f7dae3b84333fbb641e6e0f',
 'Frozen_Exp5Prescreen_Blocks_2026-09-30.csv': '2f9b00ccbfc028d203b2f45ebf1d4e95363ca5de2794f41c2db67415b8f08001',
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()
def check_input(name):
    p = find(name); got = sha256(p); ok = got == EXPECTED_SHA.get(name)
    print(f"{'OK      ' if ok else 'MISMATCH'} {name}\n         sha256={got}")
    return p
print('Input files:')
for n in list(EXPECTED_SHA)[:6]: check_input(n)

Input files:
OK       Frozen_Exp5Prescreen_Blocks_2026-07-25.csv
         sha256=bd061f5689188428b464dfab3c847491a4d731a465728facb1f989b08b3d1b42
OK       Frozen_Exp5Prescreen_Sessions_2026-07-25.csv
         sha256=00bac7096bb716e07c28c7363a5facea3e702b8d245943bf2ffc20ae41081dc3
OK       Frozen_Exp5Prescreen_Blocks_2026-09-23.csv
         sha256=ab995877735a27097b5ad28c87113bf879e35d1537bae047058bd61d82b6a863
OK       Frozen_Exp5Prescreen_Sessions_2026-09-23.csv
         sha256=3850a6ed8dc90b0bcff6cee961b4fbb84953f0588b17acdaa2e24dd3388996ea
OK       Frozen_Blocks_2026-02-10_195735.csv
         sha256=3328875750873c704abc47c872064c4878cb0f9d7f621a95dfdcc393b9f0b529
OK       Frozen_Sessions_2026-02-10_195735.csv
         sha256=6b54ae50ba65bcfaf2fe5a4f452765540dee2e1a3afd2da524418e5bc97f83b9


In [2]:
# ---- VERBATIM from Variance_Narrowing_executed_9_17_26.ipynb ----
# hurstApprox (setup cell), windowed_correlations (Part 1), windowed_r_tagged/build_nested/clustered_resample/
# run_bootstrap_exchangeable (Part 2), observed_combined_stat_p16/bootstrap_ci_p16 (Part 16e / 17b).

def hurstApprox(bits):
    """Single-scale R/S Hurst exponent. Exact port of the JS hurstApprox used in the experiment
    -- same implementation used throughout Notebook 2 and Notebook 2b."""
    if bits is None:
        return np.nan
    bits = np.asarray(bits, dtype=int)
    n = bits.size
    if n < 20:
        return np.nan
    x = np.where(bits != 0, 1.0, -1.0)
    mean = float(x.sum() / n)
    y = 0.0; minY = 0.0; maxY = 0.0; s2 = 0.0
    for i in range(n):
        d = x[i] - mean
        y += d
        if y < minY: minY = y
        if y > maxY: maxY = y
        s2 += d * d
    R = maxY - minY
    S = np.sqrt(s2 / n) if s2 > 0 else 1.0
    if not np.isfinite(S) or S == 0: S = 1.0
    ratio = (R / S) if (R / S) != 0 else 1.0
    if not np.isfinite(ratio) or ratio == 0: ratio = 1.0
    H = np.log(ratio) / np.log(n)
    if not np.isfinite(H): H = 0.5
    return float(max(0.0, min(1.0, H)))



def windowed_correlations(df_in, W, block_idx_col='block_idx'):
    r_vals = []
    for sid, g in df_in.groupby('session_id'):
        g = g.sort_values(block_idx_col)
        hs = g['hurst_subj_rough'].values; hp = g['hurst_demon_rough'].values
        n_windows = len(hs) // W
        for i in range(n_windows):
            a = hs[i*W:(i+1)*W]; b = hp[i*W:(i+1)*W]
            if np.std(a) > 0 and np.std(b) > 0:
                r_vals.append(np.corrcoef(a, b)[0, 1])
    return np.array(r_vals)


def windowed_r_tagged(df_in, W, block_idx_col='block_idx'):
    rows = []
    for sid, g in df_in.groupby('session_id'):
        g = g.sort_values(block_idx_col)
        hs = g['hurst_subj_rough'].values; hp = g['hurst_demon_rough'].values
        pid = g['participant_id'].iloc[0]
        n_windows = len(hs)//W
        for i in range(n_windows):
            a=hs[i*W:(i+1)*W]; b=hp[i*W:(i+1)*W]
            if np.std(a)>0 and np.std(b)>0:
                rows.append({'participant_id': pid, 'session_id': sid, 'r': np.corrcoef(a,b)[0,1]})
    return pd.DataFrame(rows)

def build_nested(windows_df, col='r'):
    nested = {}
    for pid, pgrp in windows_df.groupby('participant_id'):
        nested[pid] = [sgrp[col].values for _, sgrp in pgrp.groupby('session_id')]
    return nested

def clustered_resample(nested, pids, rng):
    n_pids = len(pids)
    sampled_pid_idx = rng.integers(0, n_pids, size=n_pids)
    out = []
    for idx in sampled_pid_idx:
        sessions = nested[pids[idx]]
        n_sess = len(sessions)
        sampled_sess_idx = rng.integers(0, n_sess, size=n_sess)
        for sidx in sampled_sess_idx:
            out.append(sessions[sidx])
    return np.concatenate(out)

def run_bootstrap_exchangeable(human_windows, base_pool, n_boot, rng):
    obs_ratio = human_windows['r'].values.var(ddof=1) / base_pool.var(ddof=1)
    h_nested = build_nested(human_windows)
    h_pids = list(h_nested.keys())
    n_base = len(base_pool)
    boot_ratios = np.empty(n_boot)
    for i in range(n_boot):
        h_synth = clustered_resample(h_nested, h_pids, rng)
        b_synth = rng.choice(base_pool, size=n_base, replace=True)
        boot_ratios[i] = h_synth.var(ddof=1) / b_synth.var(ddof=1)
    ci_lo, ci_hi = np.percentile(boot_ratios, [2.5, 97.5])
    return dict(obs_ratio=obs_ratio, ci_lo=ci_lo, ci_hi=ci_hi)


W_VALUES_P16 = [3, 5, 10]

def observed_combined_stat_p16(human_df, baseline_windows_by_w):
    logs = []
    for W in W_VALUES_P16:
        h = windowed_r_tagged(human_df, W)['r'].values
        b = np.concatenate(list(baseline_windows_by_w[W].values()))
        logs.append(np.log(h.var(ddof=1) / b.var(ddof=1)))
    return np.mean(logs)

def bootstrap_ci_p16(human_df, baseline_windows_by_w, n_boot, rng):
    """Participant-then-session cluster bootstrap (Part 16a terminology -- NOT a three-stage
    participant->session->window bootstrap; all windows from a selected session are kept, not
    resampled individually). Resamples the participant/session hierarchy ONCE per iteration and
    reuses that exact same resampled set of sessions to compute all three W values -- preserves
    the real dependence between W=3/5/10 (Part 16c fix). Baseline resampled independently, at
    the session level (Part 15 fix -- Baseline's participant IDs are arbitrary automated labels,
    not real clusters)."""
    pid_to_sessions = human_df.groupby('participant_id')['session_id'].unique().apply(list).to_dict()
    pids = list(pid_to_sessions.keys())
    n_pids = len(pids)
    human_sess_windows_by_w = {
        W: {sid: g['r'].values for sid, g in windowed_r_tagged(human_df, W).groupby('session_id')}
        for W in W_VALUES_P16
    }
    baseline_sids = list(baseline_windows_by_w[W_VALUES_P16[0]].keys())
    n_base = len(baseline_sids)
    draws = np.empty(n_boot)
    for i in range(n_boot):
        sampled_pid_idx = rng.integers(0, n_pids, size=n_pids)
        resampled_human_sids = []
        for idx in sampled_pid_idx:
            pid = pids[idx]
            sess_list = pid_to_sessions[pid]
            n_sess = len(sess_list)
            sess_idx = rng.integers(0, n_sess, size=n_sess)
            resampled_human_sids.extend(sess_list[s] for s in sess_idx)
        base_idx = rng.integers(0, n_base, size=n_base)
        resampled_base_sids = [baseline_sids[j] for j in base_idx]
        logs = []
        for W in W_VALUES_P16:
            h_pool = np.concatenate([human_sess_windows_by_w[W][sid] for sid in resampled_human_sids])
            b_pool = np.concatenate([baseline_windows_by_w[W][sid] for sid in resampled_base_sids])
            logs.append(np.log(h_pool.var(ddof=1) / b_pool.var(ddof=1)))
        draws[i] = np.mean(logs)
    return draws


In [3]:
# ---- loaders and shared helpers (new code) ----
W_POOLED = [3, 5, 10]          # curve-level statistic
W_SD = [5, 10, 15, 20]         # session-SD / Part 7
DECOUPLED_WINDOW = (pd.Timestamp('2026-08-19', tz='UTC'), pd.Timestamp('2026-08-28', tz='UTC'))

def load_exp5_raw(snap):
    """Blocks joined to sessions; Human + Baseline only; Hurst per stream. No exclusions yet."""
    b = pd.read_csv(find(f'Frozen_Exp5Prescreen_Blocks_{snap}.csv'))
    s = pd.read_csv(find(f'Frozen_Exp5Prescreen_Sessions_{snap}.csv'))
    b['subject_bits'] = b['subject_bits'].apply(ast.literal_eval); b['demon_bits'] = b['demon_bits'].apply(ast.literal_eval)
    d = b.merge(s[['session_id', 'condition', 'participant_id', 'createdAt']], on='session_id', how='left')
    d = d[d['condition'].isin(['human', 'baseline'])].copy()
    d['hurst_subj_rough'] = d['subject_bits'].apply(hurstApprox); d['hurst_demon_rough'] = d['demon_bits'].apply(hurstApprox)
    d['created'] = pd.to_datetime(d['createdAt'], utc=True, errors='coerce')
    return d.drop(columns=['subject_bits', 'demon_bits'])

def finish(d, drop_decoupled=False):
    """Optionally drop sessions created 2026-08-19..08-28 (Subject/PCS assignment decoupled from the QRNG call),
    then add session_count (= sessions per participant x condition), as the notebook does."""
    d = d.copy()
    if drop_decoupled:
        m = (d['created'] >= DECOUPLED_WINDOW[0]) & (d['created'] < DECOUPLED_WINDOW[1])
        d = d[~m].copy()
    v = d.groupby(['participant_id', 'condition'])['session_id'].nunique().reset_index()
    v.columns = ['participant_id', 'condition', 'session_count']
    return d.merge(v, on=['participant_id', 'condition'], how='left')

def load_exp4():
    b = pd.read_csv(find('Frozen_Blocks_2026-02-10_195735.csv')); ss = pd.read_csv(find('Frozen_Sessions_2026-02-10_195735.csv'))
    def pl(x):
        try: return ast.literal_eval(x)
        except Exception: return None
    d = b.copy(); d['parsed'] = d['trial_data'].apply(pl)
    d['subject_bits'] = d['parsed'].apply(lambda x: x.get('subject_bits') if x else None)
    d['demon_bits'] = d['parsed'].apply(lambda x: x.get('demon_bits') if x else None)
    d = d.rename(columns={'sessionId': 'session_id'}); s4 = ss.rename(columns={'sessionId': 'session_id', 'agent_class': 'condition'})
    d = d.merge(s4[['session_id', 'condition', 'participant_id']], on='session_id', how='left').dropna(subset=['subject_bits', 'demon_bits', 'condition']).copy()
    d = d[d['condition'].isin(['human', 'baseline'])].copy()
    bps = d.groupby('session_id').size(); d = d[d['session_id'].isin(bps[bps >= 25].index)].copy()
    v = d.groupby(['participant_id', 'condition'])['session_id'].nunique().reset_index(); v.columns = ['participant_id', 'condition', 'session_count']
    d = d.merge(v, on=['participant_id', 'condition'], how='left')
    d['hurst_subj_rough'] = d['subject_bits'].apply(hurstApprox); d['hurst_demon_rough'] = d['demon_bits'].apply(hurstApprox)
    return d.drop(columns=['subject_bits', 'demon_bits', 'parsed'])

RESULTS = {}   # name -> value, for the verification table at the end
def baseline_windows(base, Ws=W_POOLED):
    return {W: {sid: g['r'].values for sid, g in windowed_r_tagged(base, W).groupby('session_id')} for W in Ws}
def pooled_est(hum, bw, seed, ci=True):
    o = observed_combined_stat_p16(hum, bw)
    if not ci or hum['participant_id'].nunique() < 1: return o, (np.nan, np.nan)
    dr = bootstrap_ci_p16(hum, bw, N_BOOT_POOLED, np.random.default_rng(seed))
    return o, tuple(np.percentile(dr, [2.5, 97.5]))
def fmt(o, ci): return f"{o:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]"

print('Loading exp5-prescreen 07-25 ...'); EXP5_0725 = finish(load_exp5_raw('2026-07-25'))
print('Loading exp5-prescreen 09-23 ...'); _raw0923 = load_exp5_raw('2026-09-23')
EXP5_0923 = finish(_raw0923); EXP5_0923_DEC = finish(_raw0923, drop_decoupled=True)
print('Loading Exp4 ...'); EXP4 = load_exp4()
for name, d in [('exp5 07-25', EXP5_0725), ('exp5 09-23', EXP5_0923), ('exp5 09-23 decoupled dropped', EXP5_0923_DEC), ('Exp4', EXP4)]:
    h = d[d.condition == 'human']; b = d[d.condition == 'baseline']
    print(f"{name:<30} Human: {h.participant_id.nunique()} participants, {h.session_id.nunique()} sessions, {len(h)} blocks | Baseline: {b.session_id.nunique()} sessions")

Loading exp5-prescreen 07-25 ...


Loading exp5-prescreen 09-23 ...


Loading Exp4 ...


exp5 07-25                     Human: 12 participants, 35 sessions, 3360 blocks | Baseline: 59 sessions
exp5 09-23                     Human: 10 participants, 25 sessions, 2000 blocks | Baseline: 340 sessions
exp5 09-23 decoupled dropped   Human: 9 participants, 22 sessions, 1760 blocks | Baseline: 222 sessions
Exp4                           Human: 121 participants, 158 sessions, 4737 blocks | Baseline: 103 sessions


## 1. How much of the Human data is the PI
Share of Human sessions/blocks belonging to the PI/test accounts, before any removal. (The 08-26 snapshot is the most complete pre-deletion picture; it is used only if the file is present.)

In [4]:
rows = []
def share_row(label, d):
    h = d[d.condition == 'human']; p = h[h.participant_id.isin(PI_IDS)]; y = h[h.participant_id == Y612]
    rows.append(dict(snapshot=label, human_sessions=f"{p.session_id.nunique()} / {h.session_id.nunique()} = {100*p.session_id.nunique()/h.session_id.nunique():.1f}%",
                     human_blocks=f"{len(p)} / {len(h)} = {100*len(p)/len(h):.1f}%",
                     y612_alone=f"{y.session_id.nunique()} sess ({100*y.session_id.nunique()/h.session_id.nunique():.1f}%), {len(y)} blocks ({100*len(y)/len(h):.1f}%)"))
    return 100 * p.session_id.nunique() / h.session_id.nunique(), 100 * len(p) / len(h)
RESULTS['share_0725_sessions_pct'], RESULTS['share_0725_blocks_pct'] = share_row('exp5 07-25', EXP5_0725)
share_row('exp5 09-23 (decoupled dropped)', EXP5_0923_DEC)
if find('Frozen_Exp5Prescreen_Blocks_2026-08-26.csv', required=False):
    _d826 = finish(load_exp5_raw('2026-08-26')); share_row('exp5 08-26 (pre-deletion)', _d826)
print(pd.DataFrame(rows).to_string(index=False))
h4 = EXP4[EXP4.condition == 'human']; y4 = h4[h4.participant_id == Y612]
h45 = h4[h4.session_count >= 5]; y45 = h45[h45.participant_id == Y612]
print(f"\nExp4 y612: {y4.session_id.nunique()} of {h4.session_id.nunique()} Human sessions ({len(y4)} blocks); "
      f"{y45.session_id.nunique()} of {h45.session_id.nunique()} Human5+ sessions ({len(y45)} of {len(h45)} blocks = {100*len(y45)/len(h45):.0f}%)")
RESULTS['exp4_y612_h5_sessions'] = y45.session_id.nunique(); RESULTS['exp4_h5_sessions'] = h45.session_id.nunique()
_b4 = EXP4[(EXP4.condition == 'baseline')]; _by = _b4[_b4.participant_id == Y612]
print(f"Exp4 y612 is also a Baseline contributor: {_by.session_id.nunique()} of {_b4.session_id.nunique()} Baseline sessions -- KEPT in Baseline")

                      snapshot  human_sessions        human_blocks                          y612_alone
                    exp5 07-25 19 / 35 = 54.3% 2080 / 3360 = 61.9% 8 sess (22.9%), 1130 blocks (33.6%)
exp5 09-23 (decoupled dropped)  4 / 22 = 18.2%  320 / 1760 = 18.2%      0 sess (0.0%), 0 blocks (0.0%)

Exp4 y612: 18 of 158 Human sessions (540 blocks); 18 of 28 Human5+ sessions (540 of 840 blocks = 64%)
Exp4 y612 is also a Baseline contributor: 41 of 103 Baseline sessions -- KEPT in Baseline


## 2. exp5-prescreen pooled-variance curve estimate and PI decomposition
Statistic: mean over W=3/5/10 of log(pooled variance ratio). Rows: **A** PI accounts as separate UIDs (as published); **B** PI merged into one person; **C** PI removed from Human only; **E** PI alone (one person; CI = session bootstrap within the PI). PI contribution = A − C. Leave-one-person-out treats the PI as one person.

In [5]:
def pi_decomposition(label, d, seed=SEED_POOLED_EXP5):
    hum_all = d[d.condition == 'human'].copy(); base = d[d.condition == 'baseline'].copy()
    is_pi = hum_all.participant_id.isin(PI_IDS)
    bw = baseline_windows(base)                 # Baseline untouched
    hum_merged = hum_all.copy(); hum_merged.loc[is_pi, 'participant_id'] = PI_TAG
    hum_noPI = hum_all[~is_pi]; hum_PIonly = hum_merged[is_pi]
    print(f"\n################ {label}")
    print(f"Human: {hum_all.participant_id.nunique()} UIDs ({hum_all[is_pi].participant_id.nunique()} PI/test UIDs -> ONE person = {hum_noPI.participant_id.nunique()+1} persons); "
          f"PI sessions {hum_all[is_pi].session_id.nunique()} of {hum_all.session_id.nunique()}; PI blocks {is_pi.sum()} of {len(hum_all)} ({100*is_pi.mean():.1f}%)")
    A = pooled_est(hum_all, bw, seed); B = pooled_est(hum_merged, bw, seed); C = pooled_est(hum_noPI, bw, seed); E = pooled_est(hum_PIonly, bw, seed)
    print(f"A. PI included, separate UIDs (as published)   : {fmt(*A)}  N={hum_all.participant_id.nunique()}")
    print(f"B. PI included, merged into ONE person         : {fmt(*B)}  N={hum_merged.participant_id.nunique()}")
    print(f"C. PI removed from Human only (Baseline kept)  : {fmt(*C)}  N={hum_noPI.participant_id.nunique()}")
    print(f"E. PI ALONE (session bootstrap within PI)      : {fmt(*E)}  sessions={hum_PIonly.session_id.nunique()} blocks={len(hum_PIonly)}")
    print(f"   PI contribution = A - C = {A[0]-C[0]:+.4f}  ({100*(A[0]-C[0])/A[0]:.0f}% of the with-PI estimate; |estimate| shrinks {abs(A[0]):.4f} -> {abs(C[0]):.4f})")
    print("   per-W log(variance ratio) and share of windows:")
    for W in W_POOLED:
        wp = windowed_r_tagged(hum_PIonly, W)['r'].values; wn = windowed_r_tagged(hum_noPI, W)['r'].values; wa = np.concatenate([wp, wn])
        bvar = np.concatenate(list(bw[W].values())).var(ddof=1)
        print(f"   W={W:>2}: PI windows {len(wp)}/{len(wa)} ({100*len(wp)/len(wa):.0f}%)  all={np.log(wa.var(ddof=1)/bvar):+.4f}  PI={np.log(wp.var(ddof=1)/bvar):+.4f}  non-PI={np.log(wn.var(ddof=1)/bvar):+.4f}")
    print("   leave-one-PERSON-out (PI merged); shift = estimate_without - full:")
    loo = []
    for pz in hum_merged.participant_id.unique():
        x = hum_merged[hum_merged.participant_id != pz]
        if x.participant_id.nunique() < 1: continue
        o = observed_combined_stat_p16(x, bw); loo.append((pz, o - B[0], o, int((hum_merged.participant_id == pz).sum())))
    for pz, sh, o, nb in sorted(loo, key=lambda t: -abs(t[1])):
        print(f"     drop {('PI (all accounts)' if pz == PI_TAG else pz[:8]):<18} blocks={nb:>4}  estimate={o:+.4f}  shift={sh:+.4f}")
    return dict(A=A, B=B, C=C, E=E)

dec_0725 = pi_decomposition('exp5-prescreen 07-25 (native 80-block sessions)', EXP5_0725)
RESULTS.update({'p_0725_with': dec_0725['A'][0], 'p_0725_with_ci_lo': dec_0725['A'][1][0], 'p_0725_with_ci_hi': dec_0725['A'][1][1],
                'p_0725_merged_ci_lo': dec_0725['B'][1][0], 'p_0725_merged_ci_hi': dec_0725['B'][1][1],
                'p_0725_without': dec_0725['C'][0], 'p_0725_without_ci_lo': dec_0725['C'][1][0], 'p_0725_without_ci_hi': dec_0725['C'][1][1],
                'p_0725_PIalone': dec_0725['E'][0], 'p_0725_contrib_pct': 100 * (dec_0725['A'][0] - dec_0725['C'][0]) / dec_0725['A'][0]})
dec_0923 = pi_decomposition('exp5-prescreen 09-23 (decoupled-assignment sessions 08-19..08-28 dropped)', EXP5_0923_DEC)
RESULTS.update({'p_0923_with': dec_0923['A'][0], 'p_0923_without': dec_0923['C'][0], 'p_0923_without_ci_lo': dec_0923['C'][1][0],
                'p_0923_without_ci_hi': dec_0923['C'][1][1], 'p_0923_PIalone': dec_0923['E'][0]})
_d = dec_0923['C'][0] - dec_0725['A'][0]
print(f"\nThe published -0.0887 (07-25, with PI) -> -0.0586 (09-23, without PI) change = {_d:+.4f}: "
      f"PI removal {dec_0725['C'][0]-dec_0725['A'][0]:+.4f} + snapshot change {dec_0923['C'][0]-dec_0725['C'][0]:+.4f}")


################ exp5-prescreen 07-25 (native 80-block sessions)
Human: 12 UIDs (6 PI/test UIDs -> ONE person = 7 persons); PI sessions 19 of 35; PI blocks 2080 of 3360 (61.9%)


A. PI included, separate UIDs (as published)   : -0.0887  95% CI [-0.1804, +0.0239]  N=12
B. PI included, merged into ONE person         : -0.0887  95% CI [-0.1747, +0.0329]  N=7
C. PI removed from Human only (Baseline kept)  : -0.0523  95% CI [-0.1787, +0.0694]  N=6
E. PI ALONE (session bootstrap within PI)      : -0.1114  95% CI [-0.2127, -0.0122]  sessions=19 blocks=2080
   PI contribution = A - C = -0.0364  (41% of the with-PI estimate; |estimate| shrinks 0.0887 -> 0.0523)
   per-W log(variance ratio) and share of windows:
   W= 3: PI windows 686/1102 (62%)  all=+0.0045  PI=+0.0183  non-PI=-0.0164
   W= 5: PI windows 416/672 (62%)  all=-0.0699  PI=-0.0803  non-PI=-0.0513


   W=10: PI windows 208/336 (62%)  all=-0.2006  PI=-0.2721  non-PI=-0.0891
   leave-one-PERSON-out (PI merged); shift = estimate_without - full:


     drop PI (all accounts)  blocks=2080  estimate=-0.0523  shift=+0.0364
     drop jaFBRIlW           blocks= 240  estimate=-0.1046  shift=-0.0160
     drop 7CjD2CPK           blocks= 320  estimate=-0.0844  shift=+0.0043
     drop v7vrmX8R           blocks= 400  estimate=-0.0927  shift=-0.0041
     drop K0ZprKuH           blocks=  80  estimate=-0.0854  shift=+0.0032
     drop RSrkU8Yd           blocks= 160  estimate=-0.0901  shift=-0.0014
     drop YPPx1tgb           blocks=  80  estimate=-0.0899  shift=-0.0012



################ exp5-prescreen 09-23 (decoupled-assignment sessions 08-19..08-28 dropped)
Human: 9 UIDs (2 PI/test UIDs -> ONE person = 8 persons); PI sessions 4 of 22; PI blocks 320 of 1760 (18.2%)


A. PI included, separate UIDs (as published)   : -0.0304  95% CI [-0.1233, +0.0673]  N=9
B. PI included, merged into ONE person         : -0.0304  95% CI [-0.1341, +0.0647]  N=8
C. PI removed from Human only (Baseline kept)  : -0.0586  95% CI [-0.1635, +0.0469]  N=7
E. PI ALONE (session bootstrap within PI)      : +0.1057  95% CI [-0.0065, +0.1796]  sessions=4 blocks=320
   PI contribution = A - C = +0.0282  (-93% of the with-PI estimate; |estimate| shrinks 0.0304 -> 0.0586)
   per-W log(variance ratio) and share of windows:
   W= 3: PI windows 104/572 (18%)  all=+0.0100  PI=+0.1285  non-PI=-0.0161
   W= 5: PI windows 64/352 (18%)  all=+0.0071  PI=+0.1658  non-PI=-0.0278
   W=10: PI windows 32/176 (18%)  all=-0.1082  PI=+0.0229  non-PI=-0.1319
   leave-one-PERSON-out (PI merged); shift = estimate_without - full:


     drop PI (all accounts)  blocks= 320  estimate=-0.0586  shift=-0.0282
     drop 7CjD2CPK           blocks= 400  estimate=-0.0044  shift=+0.0260
     drop jaFBRIlW           blocks= 240  estimate=-0.0511  shift=-0.0207
     drop K0ZprKuH           blocks=  80  estimate=-0.0218  shift=+0.0086
     drop v7vrmX8R           blocks= 400  estimate=-0.0221  shift=+0.0083
     drop RSrkU8Yd           blocks= 160  estimate=-0.0266  shift=+0.0037
     drop BbqfPSNh           blocks=  80  estimate=-0.0271  shift=+0.0033
     drop YPPx1tgb           blocks=  80  estimate=-0.0303  shift=+0.0000

The published -0.0887 (07-25, with PI) -> -0.0586 (09-23, without PI) change = +0.0301: PI removal +0.0364 + snapshot change -0.0063


### 2b. Human5+ stratum and the 09-23 snapshot without the decoupled-window exclusion
`session_count` >= 5 defines Human5+. PI removed from the Human arm only.

In [6]:
print('exp5-prescreen Human5+ (pooled-variance curve estimate, W=3/5/10)')
bw_0725 = baseline_windows(EXP5_0725[EXP5_0725.condition == 'baseline'])
h5 = EXP5_0725[(EXP5_0725.condition == 'human') & (EXP5_0725.session_count >= 5)]
o, ci = pooled_est(h5, bw_0725, SEED_POOLED_EXP5); RESULTS['p_0725_h5'] = o
print(f"07-25 Human5+ with PI, {h5.participant_id.nunique()} participants, {len(h5)} blocks: {fmt(o, ci)}")
bw_0923 = baseline_windows(EXP5_0923[EXP5_0923.condition == 'baseline'])
h5 = EXP5_0923[(EXP5_0923.condition == 'human') & (EXP5_0923.session_count >= 5)]
o, ci = pooled_est(h5, bw_0923, SEED_POOLED_EXP5); RESULTS['p_0923_h5'] = o
print(f"09-23 Human5+ (no PI in it), {h5.participant_id.nunique()} participants, {len(h5)} blocks: {fmt(o, ci)}")
ha = EXP5_0923[(EXP5_0923.condition == 'human') & ~EXP5_0923.participant_id.isin(PI_IDS)]
o, ci = pooled_est(ha, bw_0923, SEED_POOLED_EXP5); RESULTS['p_0923_nodec_without'] = o
print(f"09-23 Human(all), all PI removed, decoupled sessions KEPT, {ha.participant_id.nunique()} participants: {fmt(o, ci)}")

exp5-prescreen Human5+ (pooled-variance curve estimate, W=3/5/10)


07-25 Human5+ with PI, 3 participants, 1930 blocks: -0.1218  95% CI [-0.2252, -0.0042]


09-23 Human5+ (no PI in it), 2 participants, 800 blocks: -0.0906  95% CI [-0.1894, -0.0163]


09-23 Human(all), all PI removed, decoupled sessions KEPT, 7 participants: -0.0635  95% CI [-0.1657, +0.0418]


## 3. Exp4 pooled-variance curve estimate
Human(all) and Human5+, with `y612…` and with `y612…` removed from the **Human arm only** (Baseline untouched: all Baseline sessions, including `y612…`'s own, stay in).

In [7]:
base4 = EXP4[EXP4.condition == 'baseline']; bw4 = baseline_windows(base4)
print(f"Exp4 Baseline: {base4.session_id.nunique()} sessions (kept intact)")
for label, excl in [('with y612', set()), ('y612 removed from Human only', {Y612})]:
    ha = EXP4[(EXP4.condition == 'human') & ~EXP4.participant_id.isin(excl)]; h5 = ha[ha.session_count >= 5]
    for sl, hdf in [('Human(all)', ha), ('Human5+', h5)]:
        o, ci = pooled_est(hdf, bw4, SEED_POOLED_EXP4)
        dr = bootstrap_ci_p16(hdf, bw4, N_BOOT_POOLED, np.random.default_rng(SEED_POOLED_EXP4)); c90 = np.percentile(dr, [5, 95])
        print(f"{label:<30} {sl:<11} N={hdf.participant_id.nunique():>3} sessions={hdf.session_id.nunique():>3} blocks={len(hdf):>4}  {fmt(o, ci)}  90% CI [{c90[0]:+.4f}, {c90[1]:+.4f}]")
        RESULTS[f"p_exp4_{'with' if not excl else 'without'}_{sl}"] = o

Exp4 Baseline: 103 sessions (kept intact)


with y612                      Human(all)  N=121 sessions=158 blocks=4737  -0.0914  95% CI [-0.1789, +0.0026]  90% CI [-0.1656, -0.0126]


with y612                      Human5+     N=  3 sessions= 28 blocks= 840  -0.1830  95% CI [-0.5364, -0.0506]  90% CI [-0.4646, -0.0717]


y612 removed from Human only   Human(all)  N=120 sessions=140 blocks=4197  -0.0856  95% CI [-0.1787, +0.0077]  90% CI [-0.1624, -0.0051]


y612 removed from Human only   Human5+     N=  2 sessions= 10 blocks= 300  -0.2652  95% CI [-0.7133, -0.0425]  90% CI [-0.6486, -0.0771]


## 4. Part 7: Human5+ variance ratios per W (Human5+ vs Baseline, participant-clustered bootstrap)
`run_bootstrap_exchangeable`, seed 7. The `y612…`-only row is **not** PI-free (other PI/test accounts remain). The all-PI-removed row leaves a single participant in 07-25 Human5+, so its CI is not a participant-clustered interval; read it as a point estimate.

In [8]:
def part7_table(label, hum5, base):
    print(f"--- {label}: {hum5.participant_id.nunique()} participants, {len(hum5)} blocks ---")
    out = {}
    for W in W_SD:
        res = run_bootstrap_exchangeable(windowed_r_tagged(hum5, W), windowed_correlations(base, W), N_BOOT_RATIO, np.random.default_rng(7))
        out[W] = res
        print(f"  W={W:>2}  ratio={res['obs_ratio']:.4f}  CI=[{res['ci_lo']:.3f},{res['ci_hi']:.3f}]  excludes 1: {not (res['ci_lo'] <= 1 <= res['ci_hi'])}")
    return out
base_0725 = EXP5_0725[EXP5_0725.condition == 'baseline']; h5_0725 = EXP5_0725[(EXP5_0725.condition == 'human') & (EXP5_0725.session_count >= 5)]
P7 = {}
P7['none'] = part7_table('07-25 Human5+, none removed', h5_0725, base_0725)
P7['y612'] = part7_table('07-25 Human5+, y612 only removed (NOT PI-free)', h5_0725[h5_0725.participant_id != Y612], base_0725)
P7['all'] = part7_table('07-25 Human5+, all PI/test removed', h5_0725[~h5_0725.participant_id.isin(PI_IDS)], base_0725)
P7['0923'] = part7_table('09-23 Human5+ (2 participants)', EXP5_0923[(EXP5_0923.condition == 'human') & (EXP5_0923.session_count >= 5)], EXP5_0923[EXP5_0923.condition == 'baseline'])
_ha = EXP5_0725[(EXP5_0725.condition == 'human') & ~EXP5_0725.participant_id.isin(PI_IDS)]
P7['all_human_all'] = part7_table('07-25 Human(all), all PI/test removed', _ha, base_0725)
RESULTS.update({'p7_none_w5': P7['none'][5]['obs_ratio'], 'p7_none_w10': P7['none'][10]['obs_ratio'], 'p7_y612_w5': P7['y612'][5]['obs_ratio'],
                'p7_y612_w20': P7['y612'][20]['obs_ratio'], 'p7_0923_w10': P7['0923'][10]['obs_ratio']})

--- 07-25 Human5+, none removed: 3 participants, 1930 blocks ---


  W= 5  ratio=0.8904  CI=[0.769,1.121]  excludes 1: False


  W=10  ratio=0.7758  CI=[0.641,0.939]  excludes 1: True


  W=15  ratio=0.7774  CI=[0.595,1.048]  excludes 1: False


  W=20  ratio=0.7014  CI=[0.532,0.894]  excludes 1: True
--- 07-25 Human5+, y612 only removed (NOT PI-free): 2 participants, 800 blocks ---


...

  W=20  ratio=0.7297  CI=[0.533,0.922]  excludes 1: True
--- 07-25 Human(all), all PI/test removed: 6 participants, 1280 blocks ---


  W= 5  ratio=0.9499  CI=[0.784,1.121]  excludes 1: False


  W=10  ratio=0.9148  CI=[0.745,1.134]  excludes 1: False


  W=15  ratio=0.7732  CI=[0.600,0.957]  excludes 1: True


  W=20  ratio=0.7931  CI=[0.598,1.010]  excludes 1: False


## 5. Prereg-style per-session SD construction
Human − Baseline mean session SD(r). Human: each participant's mean over their sessions, then mean over participants; Baseline: mean over sessions. Bootstrap: participant → session for Human, session for Baseline; 95% CI from the percentile method. `Z_W`/`A_narrow` use a Baseline pseudo-participant reference **constructed here** (matched to each Human participant's session count); prefer the bootstrap CIs — `A_narrow` overstates significance. PI removed from the Human arm only.

In [9]:
def sess_sds(df, W):
    w = windowed_r_tagged(df, W)
    if len(w) == 0: return pd.DataFrame(columns=['session_id', 'participant_id', 'sd'])
    g = w.groupby(['session_id', 'participant_id'])['r'].agg(['std', 'count']).reset_index()
    g = g[g['count'] >= 2].rename(columns={'std': 'sd'}); return g[['session_id', 'participant_id', 'sd']]

def sd_contrast(label, hum, base, Ws=W_SD, verbose=True):
    if verbose: print(f"\n-- {label} --   (Human: {hum.participant_id.nunique()} participants, {hum.session_id.nunique()} sessions; Baseline: {base.session_id.nunique()} sessions)")
    rows, zs = [], []
    for W in Ws:
        hs = sess_sds(hum, W); bs = sess_sds(base, W)
        if hs.participant_id.nunique() < 2 or len(bs) < 2:
            if verbose: print(f"  W={W:>2}: not computable (Human sessions with >=2 windows: {len(hs)}, Baseline: {len(bs)})"); continue
            continue
        rng = np.random.default_rng(SEED_SD + W)
        hp = {p: g.sd.values for p, g in hs.groupby('participant_id')}; pk = list(hp); bsd = bs.sd.values
        obsH = np.mean([hp[p].mean() for p in pk]); obsB = bsd.mean(); diff = obsH - obsB
        bd = np.empty(N_BOOT_SD); ref = np.empty(N_BOOT_SD)
        for i in range(N_BOOT_SD):
            sp = [pk[j] for j in rng.integers(0, len(pk), len(pk))]
            hm = np.mean([hp[p][rng.integers(0, len(hp[p]), len(hp[p]))].mean() for p in sp])
            bm = bsd[rng.integers(0, len(bsd), len(bsd))].mean(); bd[i] = hm - bm
            ref[i] = np.mean([bsd[rng.integers(0, len(bsd), len(hp[p]))].mean() for p in pk])
        ci = np.percentile(bd, [2.5, 97.5]); z = (obsH - ref.mean()) / ref.std(ddof=1); zs.append(z)
        excl = not (ci[0] <= 0 <= ci[1])
        if verbose: print(f"  W={W:>2}  Human SD={obsH:.4f} (n_part={len(pk)}, n_sess={len(hs)})  Baseline SD={obsB:.4f} (n_sess={len(bs)})  diff={diff:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]  excludes 0: {excl}  Z_W={z:+.2f}")
        rows.append(dict(set=label, W=W, human_sd=obsH, baseline_sd=obsB, diff=diff, ci_lo=ci[0], ci_hi=ci[1], excl0=excl, n_part=len(pk), n_sess=len(hs)))
    if zs and verbose: print(f"  A_narrow = {np.mean([max(0, -z) for z in zs]):.3f} over {len(zs)} W (descriptive only)")
    return rows

SD_ROWS = []
def run_sd(label, hum, base): SD_ROWS.extend(sd_contrast(label, hum, base))
for snap_label, d in [('exp5 07-25', EXP5_0725), ('exp5 09-23', EXP5_0923), ('exp5 09-23 decoupled dropped', EXP5_0923_DEC)]:
    base = d[d.condition == 'baseline']; hum = d[d.condition == 'human']
    run_sd(f"{snap_label} | with PI", hum, base)
    run_sd(f"{snap_label} | without PI (Human only)", hum[~hum.participant_id.isin(PI_IDS)], base)


-- exp5 07-25 | with PI --   (Human: 12 participants, 35 sessions; Baseline: 59 sessions)


  W= 5  Human SD=0.4884 (n_part=12, n_sess=35)  Baseline SD=0.4949 (n_sess=59)  diff=-0.0065  95% CI [-0.0494, +0.0343]  excludes 0: False  Z_W=-0.44


  W=10  Human SD=0.2902 (n_part=12, n_sess=35)  Baseline SD=0.3182 (n_sess=59)  diff=-0.0280  95% CI [-0.0646, +0.0100]  excludes 0: False  Z_W=-1.50


  W=15  Human SD=0.2167 (n_part=12, n_sess=35)  Baseline SD=0.2703 (n_sess=59)  diff=-0.0537  95% CI [-0.0886, -0.0195]  excludes 0: True  Z_W=-2.58


  W=20  Human SD=0.1745 (n_part=12, n_sess=35)  Baseline SD=0.2117 (n_sess=59)  diff=-0.0372  95% CI [-0.0693, -0.0041]  excludes 0: True  Z_W=-1.94
  A_narrow = 1.618 over 4 W (descriptive only)

-- exp5 07-25 | without PI (Human only) --   (Human: 6 participants, 16 sessions; Baseline: 59 sessions)


...

  W=20  Human SD=0.1861 (n_part=9, n_sess=22)  Baseline SD=0.2134 (n_sess=219)  diff=-0.0274  95% CI [-0.0540, +0.0020]  excludes 0: False  Z_W=-1.15
  A_narrow = 1.063 over 4 W (descriptive only)

-- exp5 09-23 decoupled dropped | without PI (Human only) --   (Human: 7 participants, 18 sessions; Baseline: 222 sessions)


  W= 5  Human SD=0.4806 (n_part=7, n_sess=18)  Baseline SD=0.4933 (n_sess=221)  diff=-0.0126  95% CI [-0.0659, +0.0316]  excludes 0: False  Z_W=-0.58


  W=10  Human SD=0.2875 (n_part=7, n_sess=18)  Baseline SD=0.3211 (n_sess=220)  diff=-0.0336  95% CI [-0.0667, +0.0012]  excludes 0: False  Z_W=-1.38


  W=15  Human SD=0.2131 (n_part=7, n_sess=18)  Baseline SD=0.2559 (n_sess=220)  diff=-0.0428  95% CI [-0.0748, -0.0106]  excludes 0: True  Z_W=-1.63


  W=20  Human SD=0.1831 (n_part=7, n_sess=18)  Baseline SD=0.2134 (n_sess=219)  diff=-0.0303  95% CI [-0.0554, -0.0035]  excludes 0: True  Z_W=-1.10
  A_narrow = 1.173 over 4 W (descriptive only)


### 5b. Exp4 session SD (Human(all) and Human5+)
W=20 is not computable in Exp4 (about 30 blocks per session = 1 window at W=20); at W=15 each session has only 2 windows, so the session SD is very noisy. Human5+ is 3 participants (2 without `y612…`); its bootstrap is unreliable.

In [10]:
base4 = EXP4[EXP4.condition == 'baseline']; hum4 = EXP4[EXP4.condition == 'human']
for lab, h in [('Exp4 Human(all) | with y612', hum4), ('Exp4 Human(all) | y612 removed (Human only)', hum4[hum4.participant_id != Y612]),
               ('Exp4 Human5+ | with y612', hum4[hum4.session_count >= 5]), ('Exp4 Human5+ | y612 removed (Human only)', hum4[(hum4.session_count >= 5) & (hum4.participant_id != Y612)])]:
    run_sd(lab, h, base4)


-- Exp4 Human(all) | with y612 --   (Human: 121 participants, 158 sessions; Baseline: 103 sessions)


  W= 5  Human SD=0.4892 (n_part=121, n_sess=158)  Baseline SD=0.4991 (n_sess=103)  diff=-0.0099  95% CI [-0.0436, +0.0234]  excludes 0: False  Z_W=-0.83


  W=10  Human SD=0.2874 (n_part=121, n_sess=158)  Baseline SD=0.3173 (n_sess=103)  diff=-0.0299  95% CI [-0.0690, +0.0107]  excludes 0: False  Z_W=-2.09


  W=15  Human SD=0.2337 (n_part=120, n_sess=157)  Baseline SD=0.2589 (n_sess=103)  diff=-0.0252  95% CI [-0.0653, +0.0133]  excludes 0: False  Z_W=-1.84
  W=20: not computable (Human sessions with >=2 windows: 0, Baseline: 0)
  A_narrow = 1.588 over 3 W (descriptive only)

-- Exp4 Human(all) | y612 removed (Human only) --   (Human: 120 participants, 140 sessions; Baseline: 103 sessions)


  W= 5  Human SD=0.4895 (n_part=120, n_sess=140)  Baseline SD=0.4991 (n_sess=103)  diff=-0.0096  95% CI [-0.0425, +0.0233]  excludes 0: False  Z_W=-0.80


...

  W=10  Human SD=0.2299 (n_part=3, n_sess=28)  Baseline SD=0.3173 (n_sess=103)  diff=-0.0874  95% CI [-0.1534, -0.0259]  excludes 0: True  Z_W=-2.38


  W=15  Human SD=0.2045 (n_part=3, n_sess=28)  Baseline SD=0.2589 (n_sess=103)  diff=-0.0544  95% CI [-0.1258, +0.0194]  excludes 0: False  Z_W=-1.57
  W=20: not computable (Human sessions with >=2 windows: 0, Baseline: 0)
  A_narrow = 2.032 over 3 W (descriptive only)

-- Exp4 Human5+ | y612 removed (Human only) --   (Human: 2 participants, 10 sessions; Baseline: 103 sessions)


  W= 5  Human SD=0.4235 (n_part=2, n_sess=10)  Baseline SD=0.4991 (n_sess=103)  diff=-0.0756  95% CI [-0.1548, -0.0016]  excludes 0: True  Z_W=-1.76


  W=10  Human SD=0.2087 (n_part=2, n_sess=10)  Baseline SD=0.3173 (n_sess=103)  diff=-0.1086  95% CI [-0.1789, -0.0391]  excludes 0: True  Z_W=-2.13


  W=15  Human SD=0.2027 (n_part=2, n_sess=10)  Baseline SD=0.2589 (n_sess=103)  diff=-0.0562  95% CI [-0.1474, +0.0427]  excludes 0: False  Z_W=-1.15
  W=20: not computable (Human sessions with >=2 windows: 0, Baseline: 0)
  A_narrow = 1.682 over 3 W (descriptive only)


### 5c. Summary table (difference; * = 95% CI excludes 0)

In [11]:
sd_df = pd.DataFrame(SD_ROWS)
sd_df['cell'] = sd_df.apply(lambda r: f"{r['diff']:+.4f}{'*' if r['excl0'] else ''}", axis=1)
summary = sd_df.pivot_table(index='set', columns='W', values='cell', aggfunc='first')
order = list(dict.fromkeys(sd_df['set'])); summary = summary.loc[order]
print(summary.to_string())
for _, r in sd_df.iterrows(): RESULTS[f"sd::{r['set']}::W{int(r['W'])}"] = r['diff']

W                                                             5         10        15        20
set                                                                                           
exp5 07-25 | with PI                                     -0.0065   -0.0280  -0.0537*  -0.0372*
exp5 07-25 | without PI (Human only)                     -0.0258   -0.0236  -0.0497*   -0.0318
exp5 09-23 | with PI                                     -0.0001   -0.0189  -0.0468*   -0.0206
exp5 09-23 | without PI (Human only)                     -0.0143  -0.0351*  -0.0431*  -0.0326*
exp5 09-23 decoupled dropped | with PI                   +0.0003   -0.0226  -0.0458*   -0.0274
exp5 09-23 decoupled dropped | without PI (Human only)   -0.0126   -0.0336  -0.0428*  -0.0303*
Exp4 Human(all) | with y612                              -0.0099   -0.0299   -0.0252       NaN
Exp4 Human(all) | y612 removed (Human only)              -0.0096   -0.0298   -0.0250       NaN
Exp4 Human5+ | with y612                          

## 6. Optional: Baseline run-mode sensitivity (Section 8 toggle)
Run mode is **not recorded** in the session documents; it is inferred from session creation times (nearest-neighbour gap between consecutive Baseline sessions, batch = 8 minutes or less). Needs the 2026-09-30 Sessions file and the 09-30 and 09-23 Blocks files. Set `RUN_MODE_CHECK = True` in the config cell to run it.

In [12]:
if not RUN_MODE_CHECK:
    print('RUN_MODE_CHECK is False -- skipped.')
else:
    S30 = pd.read_csv(check_input('Frozen_Exp5Prescreen_Sessions_2026-09-30.csv')); B30p = check_input('Frozen_Exp5Prescreen_Blocks_2026-09-30.csv')
    S30['t'] = pd.to_datetime(S30.createdAt, utc=True)
    bsess = S30[S30.condition == 'baseline'].sort_values('t').reset_index(drop=True)
    bsess['nn_gap'] = pd.concat([bsess.t.diff().dt.total_seconds() / 60, (-bsess.t.diff(-1).dt.total_seconds() / 60)], axis=1).min(axis=1)
    bsess['decoupled'] = (bsess.t >= DECOUPLED_WINDOW[0]) & (bsess.t < DECOUPLED_WINDOW[1])
    def rd(p): return pd.read_csv(p, usecols=['session_id', 'block_idx', 'subject_bits', 'demon_bits'])
    b30 = rd(B30p); have = set(b30.session_id); b23 = rd(find('Frozen_Exp5Prescreen_Blocks_2026-09-23.csv')); b23 = b23[~b23.session_id.isin(have)]
    bl = pd.concat([b30, b23]); bl = bl[bl.session_id.isin(set(bsess.session_id))].copy()
    bl['subject_bits'] = bl.subject_bits.apply(ast.literal_eval); bl['demon_bits'] = bl.demon_bits.apply(ast.literal_eval)
    bl['hurst_subj_rough'] = bl.subject_bits.apply(hurstApprox); bl['hurst_demon_rough'] = bl.demon_bits.apply(hurstApprox)
    bl = bl.merge(bsess[['session_id', 'participant_id', 't', 'nn_gap', 'decoupled']], on='session_id')
    print(f"Baseline sessions {len(bsess)}; with blocks {bl.session_id.nunique()}; batch (<=8 min) {(bsess.nn_gap<=8).sum()} of which in decoupled window {((bsess.nn_gap<=8)&bsess.decoupled).sum()}")
    d = bl[~bl.decoupled]
    print('\nBatch vs individual Baseline mean session SD(r), decoupled window excluded:')
    for W in W_SD:
        st = sess_sds(d, W).merge(d.drop_duplicates('session_id')[['session_id', 'nn_gap', 't']], on='session_id')
        a = st[st.nn_gap <= 8].sd.values; i = st[st.nn_gap > 8].sd.values; rng = np.random.default_rng(SEED_SD + W)
        dd = np.array([a[rng.integers(0, len(a), len(a))].mean() - i[rng.integers(0, len(i), len(i))].mean() for _ in range(N_BOOT_SD)])
        ci = np.percentile(dd, [2.5, 97.5])
        print(f"  W={W:>2}: batch {a.mean():.4f} (n={len(a)}) individual {i.mean():.4f} (n={len(i)}) diff {a.mean()-i.mean():+.4f} CI [{ci[0]:+.4f},{ci[1]:+.4f}]  MWU p={stats.mannwhitneyu(a, i).pvalue:.3f}")
        rho, p = stats.spearmanr(st.t.astype('int64'), st.sd); print(f"         Baseline SD vs time: rho={rho:+.3f} p={p:.3f}")
    hd = EXP5_0923_DEC[(EXP5_0923_DEC.condition == 'human') & ~EXP5_0923_DEC.participant_id.isin(PI_IDS)]
    for lab, bsel in [('Baseline ALL', d), ('Baseline INDIVIDUAL-only', d[d.nn_gap > 8]), ('Baseline BATCH-only', d[d.nn_gap <= 8])]:
        sd_contrast(f"09-23 Human, PI removed (Human only) vs {lab}", hd, bsel)

RUN_MODE_CHECK is False -- skipped.


## 7. Verification against the published values
Each published figure is compared with the value recomputed above (tolerance 5e-4; CI endpoints 2e-3). `QUICK = True` runs will FAIL the bootstrap-dependent lines by design. FAIL on a point estimate means the inputs or code differ from the published run.

In [13]:
EXPECTED = [  # (RESULTS key, published value, tolerance, note)
 ('p_0725_with', -0.0887, 5e-4, 'exp5 07-25 pooled curve, with PI'),
 ('p_0725_with_ci_lo', -0.1804, 2e-3, 'CI low (UID clusters)'), ('p_0725_with_ci_hi', 0.0239, 2e-3, 'CI high (UID clusters)'),
 ('p_0725_merged_ci_lo', -0.1747, 2e-3, 'CI low (PI merged)'), ('p_0725_merged_ci_hi', 0.0329, 2e-3, 'CI high (PI merged)'),
 ('p_0725_without', -0.0523, 5e-4, 'exp5 07-25, without PI (Human only)'),
 ('p_0725_without_ci_lo', -0.1787, 2e-3, ''), ('p_0725_without_ci_hi', 0.0694, 2e-3, ''),
 ('p_0725_PIalone', -0.1114, 5e-4, 'PI alone'), ('p_0725_contrib_pct', 41.0, 1.0, 'PI contribution, % of with-PI estimate'),
 ('share_0725_sessions_pct', 54.3, 0.1, 'PI share of Human sessions, 07-25'), ('share_0725_blocks_pct', 61.9, 0.1, 'PI share of Human blocks, 07-25'),
 ('p_0923_with', -0.0304, 5e-4, 'exp5 09-23 (decoupled dropped), with PI accounts'),
 ('p_0923_without', -0.0586, 5e-4, 'exp5 09-23, PI-free (the -0.0586)'),
 ('p_0923_without_ci_lo', -0.1635, 2e-3, ''), ('p_0923_without_ci_hi', 0.0469, 2e-3, ''), ('p_0923_PIalone', 0.1057, 5e-4, ''),
 ('p_0923_nodec_without', -0.0635, 5e-4, '09-23, decoupled kept, PI removed'),
 ('p_0725_h5', -0.1218, 5e-4, '07-25 Human5+'), ('p_0923_h5', -0.0906, 5e-4, '09-23 Human5+'),
 ('p_exp4_with_Human(all)', -0.0914, 5e-4, 'Exp4 Human(all)'), ('p_exp4_with_Human5+', -0.1830, 5e-4, 'Exp4 Human5+'),
 ('p_exp4_without_Human(all)', -0.0856, 5e-4, 'Exp4 Human(all), y612 out of Human only'), ('p_exp4_without_Human5+', -0.2652, 5e-4, 'Exp4 Human5+, y612 out of Human only'),
 ('exp4_y612_h5_sessions', 18, 0, 'Exp4 y612 sessions in Human5+'), ('exp4_h5_sessions', 28, 0, 'Exp4 Human5+ sessions'),
 ('p7_none_w5', 0.8904, 5e-4, 'Part 7 ratio W=5 none removed'), ('p7_none_w10', 0.7758, 5e-4, 'W=10'),
 ('p7_y612_w5', 1.0255, 5e-4, 'y612-only W=5'), ('p7_y612_w20', 0.7144, 5e-4, 'y612-only W=20'), ('p7_0923_w10', 0.8156, 5e-4, '09-23 Human5+ W=10'),
 ('sd::exp5 07-25 | with PI::W15', -0.0537, 5e-4, 'session SD 07-25 with PI W=15'), ('sd::exp5 07-25 | with PI::W20', -0.0372, 5e-4, 'W=20'),
 ('sd::exp5 07-25 | without PI (Human only)::W15', -0.0497, 5e-4, '07-25 without PI W=15'), ('sd::exp5 07-25 | without PI (Human only)::W20', -0.0318, 5e-4, 'W=20'),
 ('sd::exp5 09-23 | without PI (Human only)::W10', -0.0351, 5e-4, '09-23 without PI W=10'), ('sd::exp5 09-23 | without PI (Human only)::W15', -0.0431, 5e-4, 'W=15'),
 ('sd::exp5 09-23 decoupled dropped | without PI (Human only)::W15', -0.0428, 5e-4, 'decoupled dropped W=15'),
 ('sd::Exp4 Human(all) | with y612::W10', -0.0299, 5e-4, 'Exp4 session SD W=10'), ('sd::Exp4 Human5+ | with y612::W5', -0.0659, 5e-4, 'Exp4 Human5+ W=5'),
 ('sd::Exp4 Human5+ | y612 removed (Human only)::W10', -0.1086, 5e-4, 'Exp4 Human5+ y612 out W=10'),
]
rows = []
for key, pub, tol, note in EXPECTED:
    got = RESULTS.get(key)
    ok = (got is not None) and abs(got - pub) <= tol
    rows.append(dict(status='PASS' if ok else ('SKIP' if got is None else 'FAIL'), published=pub, recomputed=(None if got is None else round(float(got), 4)), item=key, note=note))
ver = pd.DataFrame(rows); print(ver.to_string(index=False))
print(f"\nPASS {sum(ver.status=='PASS')} / FAIL {sum(ver.status=='FAIL')} / SKIP {sum(ver.status=='SKIP')}   (total runtime {(time.time()-T0)/60:.1f} min)")

status  published  recomputed                                                            item                                             note
  PASS    -0.0887     -0.0887                                                     p_0725_with                 exp5 07-25 pooled curve, with PI
  PASS    -0.1804     -0.1804                                               p_0725_with_ci_lo                            CI low (UID clusters)
  PASS     0.0239      0.0239                                               p_0725_with_ci_hi                           CI high (UID clusters)
  PASS    -0.1747     -0.1747                                             p_0725_merged_ci_lo                               CI low (PI merged)
  PASS     0.0329      0.0329                                             p_0725_merged_ci_hi                              CI high (PI merged)
  PASS    -0.0523     -0.0523                                                  p_0725_without              exp5 07-25, without PI (Human only)

## 8. Notes that travel with these numbers
- Superseded, do **not** quote: Exp4 `y612…` removed from **both** arms (−0.1116 Human(all), −0.2911 Human5+); exp5 07-25/09-23 "−0.0670"/"−0.0624" (PI removed from Baseline too); any session-SD table that trimmed Baseline.
- The pooled statistic and the session-SD construction are different quantities; the pooled pilot estimates do not directly estimate the session-SD quantity.
- The 09-23 Blocks file is pre-deletion (stale) but Human rows are restricted to sessions in the post-deletion Sessions file; the 07-25 snapshot is the one with the PI's `y612…` sessions present.
- Bootstraps over 2–3 participants (Human5+; 07-25 Human5+ without the PI = 1 participant) are not reliable interval estimates.